In [21]:
import os

import pandas as pd
import requests
from sqlalchemy import create_engine

In [22]:
API_KEY = "waseet-demo-key"
DB_URL = os.environ.get("WASEET_DB_URL",
                        "postgresql+psycopg2://de:de@localhost:5442/waseet")

engine = create_engine(DB_URL)

In [23]:
import time
import requests
def get_with_retry(url, headers, params=None, attempts=4):
    wait = 1
    for attempt in range(attempts):
        reply = requests.get(url, params=params, headers=headers, timeout=5)
        
        if reply.status_code != 429 and not (500 <= reply.status_code < 600):
            return reply
            
        print(f"  Transient error {reply.status_code} on attempt {attempt + 1} - waiting {wait} seconds")
        time.sleep(wait)
        wait = wait * 2
        
    return reply
    raise NotImplementedError

In [24]:
def fetch_all_customers(base_url):
    """TODO. Page until has_more is False. Return a list of dicts.

    Drive the loop on what the API tells you, not on a page count you worked out
    yourself - the second one is right until the day the data grows.
    """
    headers = {"X-API-Key": os.environ.get("WASEET_API_KEY", "waseet-demo-key")}
    page = 1
    customers = []

    while True:
        reply = get_with_retry(base_url + "/customers", headers, {"page": page})
        reply.raise_for_status()
        body = reply.json()
        customers.extend(body.get("customers", body.get("observations", [])))
        if not body.get("has_more", False):
            break
        page += 1

    return customers


In [26]:
import psycopg2
from psycopg2.extras import execute_values
def load_customers(records, conn_string="postgresql://de:de@localhost:5442/waseet"):
    """Land them in the customers table.

    Idempotent: running this twice must not double the dimension, and must not
    fail either. The table has a primary key, which rules out a plain append.
    """
    if not records:
        print("No records to load.")
        return

    rows = [
        (
            r.get("customer_id"),
            r.get("customer_name"),
            r.get("segment"),
            r.get("city"),
            r.get("signup_date"),
            r.get("credit_limit")
        )
        for r in records
    ]

    query = """
        INSERT INTO customers (customer_id, customer_name, segment, city, signup_date, credit_limit)
        VALUES %s
        ON CONFLICT (customer_id) DO UPDATE SET
            customer_name = EXCLUDED.customer_name,
            segment = EXCLUDED.segment,
            city = EXCLUDED.city,
            signup_date = EXCLUDED.signup_date,
            credit_limit = EXCLUDED.credit_limit;
    """
    conn = psycopg2.connect(conn_string)
    try:
        with conn.cursor() as cur:
            execute_values(cur, query, rows)
        conn.commit()
        print(f"Successfully loaded/upserted {len(rows)} customer records into PostgreSQL.")
    except Exception as e:
        conn.rollback()
        print(f"Error loading customers: {e}")
        raise e
    finally:
        conn.close()
    #raise NotImplementedError

In [27]:
if __name__ == "__main__":
    import sys
    sys.path.insert(0, "../api")
    from waseet_api import start_api

    base_url = start_api()
    print("API on", base_url)

    records = fetch_all_customers(base_url)
    print("fetched", len(records), "customers")

    load_customers(records)
    print("loaded")
    

API on http://127.0.0.1:59725
fetched 200 customers
Successfully loaded/upserted 200 customer records into PostgreSQL.
loaded
